# Week 1 — Unpacking the black box

An n8n Anthropic node is a thin layer over one HTTP request. This week you make that request yourself, read every field of the response, and then see what LangChain adds on top.

**Core:** steps 1–5 and the self-check. **Minimum viable week:** steps 1–3 and the `ask` self-check.
**Stretch:** your own `TruncatedOutputError` class; a timeout and retry with backoff in `ask`.

Habit for the whole course: **predict before you run.** Where a cell says *Predict*, write your prediction in the text cell below it first.

In [ ]:
# Setup (run first)
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "course-tools @ git+https://github.com/YOUR-GITHUB-ORG/harness-starter@v1#subdirectory=course-tools"],
                   check=True)
    from google.colab import drive
    drive.mount("/content/drive")

import course_tools as ct
env = ct.bootstrap()
MODEL = ct.MODELS["haiku"]

## Step 1 — One raw call

Call Claude with the Anthropic SDK and print the **whole** response object. Five things go in: the model, a system prompt, the messages, `max_tokens`, and (optionally) `temperature`.

In [ ]:
import anthropic

client = anthropic.Anthropic()

# TODO: call client.messages.create(...) with model=MODEL, max_tokens=300, a short system prompt,
# and one user message asking "In two sentences, what is a three-way match?"
response = ...
print(response.model_dump_json(indent=2))

## Step 2 — Predict, then print

*Predict:* what are the type and an example value of each of these?

- `response.content[0].text`
- `response.stop_reason`
- `response.usage`

Your predictions:

- ...

In [ ]:
print(repr(response.content[0].text[:80]))
print(response.stop_reason)
print(response.usage)

This is why `$json.output` and `output[0].content[0].text` confused people in n8n: the reply is a **list of content blocks**, and text is only one kind of block. Next week you will see image blocks; in Week 3, tool-use blocks.

## Step 3 — `ask()`, with a guard against truncation

When the reply hits `max_tokens`, the API still returns successfully: the text is simply cut off and `stop_reason` is `"max_tokens"`. A harness must never pass a truncated answer on as if it were complete.

Write `ask(prompt, system=None, max_tokens=1024)` that returns the reply text and raises `TruncatedOutputError` if the reply was cut off.

In [ ]:
class TruncatedOutputError(Exception):
    """The model stopped because it hit max_tokens: the output is incomplete."""


def ask(prompt: str, system: str | None = None, max_tokens: int = 1024) -> str:
    # TODO 1: build the arguments for client.messages.create (only pass system if it is not None)
    # TODO 2: make the call
    # TODO 3: if response.stop_reason == "max_tokens", raise TruncatedOutputError
    # TODO 4: return the reply text
    raise NotImplementedError


print(ask("Name three documents in procure-to-pay."))

In [ ]:
try:
    ask("Explain in detail how a supplier invoice is processed, step by step.", max_tokens=20)
except TruncatedOutputError as e:
    print("Caught:", e)

## Step 4 — The same call through LangChain

`init_chat_model` wraps the same HTTP request. *Predict:* what type will `msg` be, and where are the text, the stop reason and the token counts now?

In [ ]:
from langchain.chat_models import init_chat_model

# TODO: create llm with init_chat_model(MODEL, model_provider="anthropic", max_tokens=300)
# and invoke it with the same system prompt and question as step 1.
llm = ...
msg = ...
print(type(msg))
print(msg.content[:80])
print(msg.response_metadata.get("stop_reason"))
print(msg.usage_metadata)

What did the abstraction add, and what did it hide? Write two lines:

- Added: ...
- Hidden: ...

## Step 5 — `cost_of()`

Input and output tokens are priced differently. The course price table (USD per million tokens) is below; always re-check prices on the Anthropic site.

Write `cost_of(usage, model)` that works for both the SDK's `response.usage` object **and** LangChain's `msg.usage_metadata` dict.

In [ ]:
ct.PRICES_PER_MTOK

In [ ]:
def cost_of(usage, model: str) -> float:
    # TODO 1: get input and output tokens (usage may be an object with attributes, or a dict)
    # TODO 2: look up (price_in, price_out) in ct.PRICES_PER_MTOK[model]
    # TODO 3: return the cost in USD (prices are per million tokens)
    raise NotImplementedError


print(f"Step 1 call: USD {cost_of(response.usage, MODEL):.6f}")
print(f"LangChain call: USD {cost_of(msg.usage_metadata, MODEL):.6f}")

## Resource meter

From this week on, every stage reports tokens and cost. Extrapolate: what would the step 1 call cost at 10,000 calls a month on Haiku, and on Opus?

In [ ]:
per_call = cost_of(response.usage, MODEL)
print(f"Haiku: USD {per_call * 10_000:,.2f} a month")
print(f"Opus:  USD {cost_of(response.usage, ct.MODELS['opus']) * 10_000:,.2f} a month")

## Break it

Open `week-01/break_it.ipynb`. Someone printed an API key into a notebook. **First rotate the key** in the Anthropic Console (anything that reached Git history must be treated as public), then find every place it leaked and remove it.

## Self-check

Runs offline against a fake model, so it costs nothing. Add `live=True` to repeat it against Claude.

In [ ]:
ct.selfcheck(1)